In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('../data/sales_clean.csv',parse_dates=['InvoiceDate'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1007910 entries, 0 to 1007909
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007910 non-null  int64         
 1   StockCode    1007910 non-null  str           
 2   Description  1007910 non-null  str           
 3   Quantity     1007910 non-null  int64         
 4   InvoiceDate  1007910 non-null  datetime64[us]
 5   Price        1007910 non-null  float64       
 6   Customer ID  779423 non-null   float64       
 7   Country      1007910 non-null  str           
 8   Revenue      1007910 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(3)
memory usage: 69.2 MB


In [3]:
missing = df["Customer ID"].isna().sum()
print(f"Rows without customers: {missing} ({missing/len(df):.1%})")
print(df["InvoiceDate"].min(),df["InvoiceDate"].max())
print(df["Customer ID"].nunique())

Rows without customers: 228487 (22.7%)
2009-12-01 07:45:00 2011-12-09 12:50:00
5878


In [4]:
df=df.dropna(subset=["Customer ID"]) 
df['Customer ID']=df['Customer ID'].astype(int)   
print(df.shape,df['Customer ID'].nunique())

(779423, 9) 5878


In [5]:
cutoff=pd.Timestamp("2011-09-09")
windows_end= cutoff +pd.Timedelta(days=90)

history = df[df['InvoiceDate']<cutoff]
window = df[(df['InvoiceDate']>=cutoff) & (df['InvoiceDate']<windows_end)]

print(history['InvoiceDate'].max())
print(window['InvoiceDate'].min(), window["InvoiceDate"].max())

2011-09-08 19:58:00
2011-09-09 08:48:00 2011-12-07 17:25:00


In [6]:
bought_in_window = window['Customer ID']
order_counts = history.groupby('Customer ID')['Invoice'].nunique()
print(order_counts.head())

Customer ID
12346    11
12347     6
12348     4
12349     3
12350     1
Name: Invoice, dtype: int64


In [7]:
regular_counts = order_counts[order_counts>=5]
regular_customers = regular_counts.index
print(len(regular_customers))



1798


In [8]:
churned = (~regular_customers.isin(bought_in_window)).astype(int)
print(churned[:5],churned.sum(),len(churned))

[1 0 0 0 0] 539 1798


In [9]:
labels = pd.DataFrame({"churned":churned},index=regular_customers)
labels.head()


,churned
Customer ID,
12346,1
12347,0
12352,0
12356,0
12359,0


In [10]:
print(labels.shape, labels["churned"].mean())

(1798, 1) 0.29977753058954393


In [12]:
history.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [18]:
last_order=history.groupby("Customer ID")["InvoiceDate"].max()
recency = (cutoff - last_order).dt.days
print(recency.head())
print(recency.min(),recency.max())


Customer ID
12346    437
12347     37
12348    156
12349    315
12350    218
Name: InvoiceDate, dtype: int64
0 646


In [20]:
recency = recency.loc[regular_customers]
recency.shape


(1798,)

In [22]:
print(recency.min(),recency.max())

0 599


In [23]:
frequency = order_counts.loc[regular_customers]
print(frequency.shape,frequency.min())

(1798,) 5


In [25]:
features = pd.DataFrame({"recency":recency,"frequency":frequency})
print(features.head())
print(features.shape)
print(features.isna().sum())

             recency  frequency
Customer ID                    
12346            437         11
12347             37          6
12352            170          7
12356            153          5
12359             97          9
(1798, 2)
recency      0
frequency    0
dtype: int64


In [27]:
monetary = history.groupby('Customer ID')['Revenue'].sum()
monetary = monetary.loc[regular_customers]
print(monetary.head())
print(f"the shape is {monetary.shape}")


Customer ID
12346     372.86
12347    3402.39
12352    1905.61
12356    6313.38
12359    6059.09
Name: Revenue, dtype: float64
the shape is (1798,)


In [33]:
features['monetary']=monetary
print(features.shape)
print(f"the number of null values are:{features['monetary'].isna().sum()}")

(1798, 3)
the number of null values are:0


In [35]:
features['avg_order']= features['monetary']/features['frequency']
print(features['avg_order'].describe())

count    1798.000000
mean      407.694422
std       409.176985
min        11.161333
25%       214.786979
50%       316.499857
75%       454.704712
max      6819.052000
Name: avg_order, dtype: float64


In [34]:
features.head()

,recency,frequency,monetary
Customer ID,,,
12346,437,11,372.86
12347,37,6,3402.39
12352,170,7,1905.61
12356,153,5,6313.38
12359,97,9,6059.09
